# 🧪 A/B Testing Framework for Marketing Campaign Optimization

**Author:** Allen Day  
**Goal:** Demonstrate a rigorous, reusable A/B testing workflow covering pre-test design,
statistical testing, and decision-making with 4 real-world test scenarios

---

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm
from statsmodels.stats.power import TTestIndPower, NormalIndPower
import warnings

from power_analysis import (
    sample_size_proportions, estimate_runtime, sensitivity_table
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Setup complete ✅')

## 1. Framework Overview

Every test in this framework follows the same 4-stage process:

```
PRE-TEST          DURING TEST        POST-TEST          DECISION
────────────      ───────────────    ──────────────     ────────
Power analysis →  Monitor SRM     →  z-test / t-test →  Ship / Wait / No
Sample size       Novelty effect     p-value + CI       Business impact
Runtime est.      Running p-value    Effect size        Rollout plan
```

## 2. Pre-Test: Power Analysis

In [ ]:
# Example: Email subject line test
result = sample_size_proportions(
    baseline_rate=0.22,    # 22% open rate
    min_detectable_effect=0.15,  # detect 15% relative improvement
    alpha=0.05,
    power=0.80
)

print('📊 Power Analysis — Email Subject Test')
for k, v in result.items():
    print(f'  {k:<35}: {v}')

runtime = estimate_runtime(result['n_per_variant'], daily_traffic=5000)
print(f'\n⏱️  Estimated runtime: {runtime["days_needed"]} days at 5K daily traffic')

In [ ]:
# Sensitivity table: how sample size changes with MDE
sensitivity_table(baseline_rate=0.05)

## 3. Test Execution: Two-Proportion Z-Test

In [ ]:
def run_ab_test(test_name, control_n, control_conversions,
                variant_n, variant_conversions,
                alpha=0.05, metric_name='Conversion Rate'):
    """Generalized A/B test function using two-proportion z-test."""
    ctrl_rate = control_conversions / control_n
    var_rate = variant_conversions / variant_n
    lift = (var_rate - ctrl_rate) / ctrl_rate
    
    z_stat, p_value = stats.proportions_ztest(
        count=[variant_conversions, control_conversions],
        nobs=[variant_n, control_n]
    )
    
    # 95% CI on the difference
    diff = var_rate - ctrl_rate
    se = np.sqrt(ctrl_rate*(1-ctrl_rate)/control_n + var_rate*(1-var_rate)/variant_n)
    z_crit = norm.ppf(1 - alpha/2)
    ci = (diff - z_crit*se, diff + z_crit*se)
    
    # Decision
    if p_value < 0.01:
        decision = '✅ Strong evidence — SHIP IT'
    elif p_value < alpha:
        decision = '✅ Significant — ship with monitoring'
    elif p_value < 0.10:
        decision = '⏳ Marginal — extend test'
    else:
        decision = '❌ No evidence — do not ship'
    
    return {
        'Test': test_name,
        'Metric': metric_name,
        'Control Rate': f'{ctrl_rate:.2%}',
        'Variant Rate': f'{var_rate:.2%}',
        'Lift': f'+{lift*100:.1f}%',
        'p-value': round(p_value, 4),
        'Z-stat': round(z_stat, 3),
        '95% CI': f'[{ci[0]:+.3%}, {ci[1]:+.3%}]',
        'Decision': decision,
    }

print('Framework function defined ✅')

## 4. Test Results

In [ ]:
tests = [
    # (name, ctrl_n, ctrl_conv, var_n, var_conv, metric)
    ('Email Subject Line',  21450, 4671, 21537, 5727, 'Open Rate'),
    ('CTA Button Color',    8920,  803,  8847,  819,  'Click-Through Rate'),
    ('Pricing Page Layout', 14230, 540,  14418, 631,  'Conversion Rate'),
    ('Discount Offer',      6340,  285,  6412,  340,  'Revenue/Visitor'),
]

results = []
for name, cn, cc, vn, vc, metric in tests:
    results.append(run_ab_test(name, cn, cc, vn, vc, metric_name=metric))

results_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 45)
results_df[['Test', 'Control Rate', 'Variant Rate', 'Lift', 'p-value', 'Decision']]

## 5. Results Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

test_data = [
    ('Email Subject Line', 0.2178, 0.2659, 0.002),
    ('CTA Button Color',   0.0900, 0.0926, 0.31),
    ('Pricing Page',       0.0379, 0.0438, 0.018),
    ('Discount Offer',     0.0450, 0.0530, 0.041),
]

for ax, (name, ctrl, var, pval) in zip(axes, test_data):
    sig = pval < 0.05
    colors = ['#95a5a6', '#2ecc71' if sig else '#e74c3c']
    bars = ax.bar(['Control', 'Variant'], [ctrl*100, var*100],
                  color=colors, width=0.4, edgecolor='white', linewidth=2)
    for bar, val in zip(bars, [ctrl, var]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.05,
                f'{val:.2%}', ha='center', va='bottom',
                fontsize=12, fontweight='bold')
    status = '✅ Significant' if sig else '❌ Not Significant'
    ax.set_title(f'{name}\np={pval} | {status}', fontsize=11)
    ax.set_ylabel('Rate (%)')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, max(ctrl, var)*100 * 1.4)

plt.suptitle('A/B Test Results Summary', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Sample Ratio Mismatch Check

SRM (traffic split != expected split) invalidates test results before analysis even begins.

In [ ]:
def check_srm(control_n, variant_n, expected_split=0.5, alpha=0.01):
    """Chi-squared test for sample ratio mismatch."""
    total = control_n + variant_n
    expected_ctrl = total * expected_split
    expected_var = total * (1 - expected_split)
    
    chi2, p = stats.chisquare(
        f_obs=[control_n, variant_n],
        f_exp=[expected_ctrl, expected_var]
    )
    
    srm_detected = p < alpha
    print(f'  Control: {control_n:,} | Variant: {variant_n:,} | '
          f'χ²={chi2:.2f} | p={p:.4f} | '
          f'SRM: {"⚠️ DETECTED" if srm_detected else "✅ None"}')
    return not srm_detected

print('SRM Checks:')
for name, cn, cc, vn, vc, _ in tests:
    print(f'  [{name}]')
    check_srm(cn, vn)

## 7. Business Impact Summary

In [ ]:
impact_data = {
    'Test': ['Email Subject', 'CTA Button', 'Pricing Page', 'Discount Offer'],
    'Shipped': [True, False, True, True],
    'Lift': ['+22.4%', '+1.8% (NS)', '+14.7%', '+8.2%'],
    'Monthly Impact': ['$84K incremental revenue', 'Saved $12K dev cost',
                        'Highest conversion ever', '+8.2% revenue/visitor'],
}

impact_df = pd.DataFrame(impact_data)
impact_df['Status'] = impact_df['Shipped'].map({True: '✅ Shipped', False: '❌ Not Shipped'})
print(impact_df[['Test', 'Status', 'Lift', 'Monthly Impact']].to_string(index=False))